# Notebook 3 — Loss Given Default Modelling
## Mortgage Credit Risk Modelling · Freddie Mac Single-Family Loan Dataset

LGD = the fraction of outstanding balance that is not recovered after a default.
This notebook covers:
- LGD distribution analysis
- Four competing models (FRM, Splines, Random Forest, XGBoost)
- Actual vs predicted scatter plots and residual analysis
- Model comparison and practical implications for capital calculation

**Prerequisites:** Run `01_data_preprocessing.py` and `04_lgd_models.py` first.


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

plt.rcParams.update({
    "figure.facecolor": "#0F1117", "axes.facecolor": "#0F1117",
    "axes.edgecolor":   "#2D3748", "axes.labelcolor": "#E2E8F0",
    "xtick.color":      "#A0AEC0", "ytick.color":     "#A0AEC0",
    "text.color":       "#E2E8F0", "grid.color":      "#1A2035",
    "legend.facecolor": "#1A2035", "legend.edgecolor":"#2D3748",
    "font.family":      "monospace", "figure.dpi":    120,
})
ACCENT  = "#38BDF8"; DANGER  = "#F87171"
SUCCESS = "#34D399"; WARN    = "#FBBF24"

PROC_DIR = Path("data/processed")
TARGET   = "lgd"

lgd_train = pd.read_parquet(PROC_DIR / "lgd_train.parquet")
lgd_oos   = pd.read_parquet(PROC_DIR / "lgd_oos.parquet")
lgd_oot   = pd.read_parquet(PROC_DIR / "lgd_oot.parquet")

print(f"LGD Train: {len(lgd_train):,}  OOS: {len(lgd_oos):,}  OOT: {len(lgd_oot):,}")
print(f"\nLGD Statistics (training set):")
print(lgd_train[TARGET].describe().round(4).to_string())


## 1. LGD Distribution Analysis

LGD is bounded to [0, 1] and typically bimodal — a spike near 0 (full recovery)
and another near 1 (total loss). This shape motivates the Fractional Response Model.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("LGD Distribution Analysis", fontsize=13,
             color="#E2E8F0", fontweight="bold")

# 1. Overall histogram
ax = axes[0]
all_lgd = pd.concat([lgd_train[TARGET], lgd_oos[TARGET], lgd_oot[TARGET]])
ax.hist(all_lgd, bins=50, color=ACCENT, alpha=0.8, edgecolor="none",
        density=True, label=f"n = {len(all_lgd):,}")
ax.axvline(all_lgd.mean(),   color=WARN,    linewidth=2,
           linestyle="--", label=f"Mean = {all_lgd.mean():.3f}")
ax.axvline(all_lgd.median(), color=SUCCESS,  linewidth=2,
           linestyle=":",  label=f"Median = {all_lgd.median():.3f}")
ax.set_xlabel("LGD", fontsize=10)
ax.set_ylabel("Density", fontsize=10)
ax.set_title("Overall LGD Distribution", color="#CBD5E1")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)

# 2. LGD by channel
if "channel" in lgd_train.columns:
    ax = axes[1]
    channel_map = {"R": "Retail", "B": "Broker", "C": "Correspondent",
                   "T": "TPO", "nan": "Unknown"}
    for ch, grp in lgd_train.groupby("channel"):
        label = channel_map.get(str(ch), str(ch))
        ax.hist(grp[TARGET], bins=30, alpha=0.5, density=True, label=label)
    ax.set_xlabel("LGD", fontsize=10)
    ax.set_ylabel("Density", fontsize=10)
    ax.set_title("LGD by Origination Channel", color="#CBD5E1")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2)

# 3. LGD vs CLTV scatter
if "orig_cltv" in lgd_train.columns:
    ax = axes[2]
    sample = lgd_train.sample(min(2000, len(lgd_train)), random_state=42)
    scatter = ax.scatter(sample["orig_cltv"], sample[TARGET],
                         alpha=0.35, s=12, c=sample[TARGET],
                         cmap="RdYlGn_r", vmin=0, vmax=1)
    plt.colorbar(scatter, ax=ax, label="LGD")
    ax.set_xlabel("Combined LTV at Origination (%)", fontsize=10)
    ax.set_ylabel("LGD", fontsize=10)
    ax.set_title("LGD vs CLTV — Higher LTV → Higher Loss", color="#CBD5E1")
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("data/figures/lgd_distribution_analysis.png",
            dpi=150, bbox_inches="tight", facecolor="#0F1117")
plt.show()


## 2. Model Comparison — Metrics Table

In [ ]:
metrics_path = PROC_DIR / "lgd_metrics.csv"

if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)

    # Pivot for clean display
    oos_metrics = metrics[metrics["split"] == "OOS"].set_index("model")
    oot_metrics = metrics[metrics["split"] == "OOT"].set_index("model")

    print("=" * 60)
    print("LGD MODEL COMPARISON — OOS")
    print("=" * 60)
    if not oos_metrics.empty:
        display_cols = [c for c in ["rmse", "mae", "r2", "bias"] if c in oos_metrics.columns]
        print(oos_metrics[display_cols].round(4).to_string())

    print("\n" + "=" * 60)
    print("LGD MODEL COMPARISON — OOT")
    print("=" * 60)
    if not oot_metrics.empty:
        print(oot_metrics[display_cols].round(4).to_string())

    # Bar chart comparison
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("LGD Model Comparison", fontsize=13,
                 color="#E2E8F0", fontweight="bold")

    palette = [ACCENT, WARN, SUCCESS, DANGER]
    for ax, (split_label, split_data) in zip(axes, [
        ("OOS", oos_metrics), ("OOT", oot_metrics)
    ]):
        if split_data.empty:
            ax.text(0.5, 0.5, f"No {split_label} data",
                    ha="center", va="center", transform=ax.transAxes, color="#A0AEC0")
            continue
        models = split_data.index.tolist()
        rmse_vals = split_data["rmse"].tolist() if "rmse" in split_data else []
        bars = ax.bar(models, rmse_vals, color=palette[:len(models)],
                      edgecolor="none", alpha=0.85, width=0.55)
        for bar, val in zip(bars, rmse_vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                    f"{val:.4f}", ha="center", va="bottom", fontsize=10,
                    color="#E2E8F0", fontweight="bold")
        ax.set_title(f"RMSE ({split_label})", color="#CBD5E1", fontsize=11)
        ax.set_ylabel("RMSE (lower = better)", fontsize=10)
        ax.grid(True, axis="y", alpha=0.3)
        ax.set_ylim(0, max(rmse_vals) * 1.3 if rmse_vals else 1)

    plt.tight_layout()
    plt.savefig("data/figures/lgd_model_comparison.png",
                dpi=150, bbox_inches="tight", facecolor="#0F1117")
    plt.show()
else:
    print("LGD metrics not found. Run 04_lgd_models.py first.")


## 3. Actual vs Predicted — Calibration Check

In [ ]:
pred_path = PROC_DIR / "lgd_predictions.csv"

if pred_path.exists():
    preds = pd.read_csv(pred_path)
    oos_preds = preds[preds["split"] == "oos"]

    pred_cols = [c for c in preds.columns if c.endswith("_pred")]
    model_names = [c.replace("_pred", "").replace("_", " ").title()
                   for c in pred_cols]

    fig, axes = plt.subplots(1, len(pred_cols), figsize=(5 * len(pred_cols), 5))
    if len(pred_cols) == 1:
        axes = [axes]

    fig.suptitle("Actual vs Predicted LGD — OOS",
                 fontsize=13, color="#E2E8F0", fontweight="bold")

    for ax, col, model_name in zip(axes, pred_cols, model_names):
        if col not in oos_preds.columns:
            continue
        y_true = oos_preds["actual_lgd"]
        y_pred = oos_preds[col]
        mask = y_true.notna() & y_pred.notna()
        y_true, y_pred = y_true[mask], y_pred[mask]
        if len(y_true) == 0:
            continue

        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        r2   = r2_score(y_true, y_pred)
        bias = (y_pred - y_true).mean()

        ax.scatter(y_true, y_pred, alpha=0.4, s=15, color=ACCENT,
                   edgecolors="none")
        ax.plot([0, 1], [0, 1], "r--", linewidth=1.5,
                label="Perfect calibration")
        # Trend line
        z = np.polyfit(y_true, y_pred, 1)
        p = np.poly1d(z)
        xx = np.linspace(0, 1, 100)
        ax.plot(xx, p(xx), color=WARN, linewidth=1.5, linestyle="-.",
                label="Fitted trend")

        ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
        ax.set_xlabel("Actual LGD",    fontsize=9)
        ax.set_ylabel("Predicted LGD", fontsize=9)
        ax.set_title(f"{model_name}\nRMSE={rmse:.3f}  R²={r2:.3f}  Bias={bias:+.3f}",
                     color="#CBD5E1", fontsize=10)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.savefig("data/figures/lgd_actual_vs_pred_oos.png",
                dpi=150, bbox_inches="tight", facecolor="#0F1117")
    plt.show()
else:
    print("LGD predictions not found. Run 04_lgd_models.py first.")


## 4. Expected Loss = PD × LGD — Portfolio Implications

The ultimate output of a credit risk model is **Expected Credit Loss (ECL)**:

```
ECL = PD × LGD × EAD
```

where EAD (Exposure at Default) is the outstanding balance.


In [ ]:
# Illustrate ECL calculation on the LGD dataset if PD scores available
xgb_res_path = PROC_DIR / "pd_xgb_results.csv"

if pred_path.exists() and xgb_res_path.exists():
    # Merge PD scores with LGD predictions on common loans
    # (This is illustrative — in production PD applies at each reporting date,
    #  LGD applies at the defaulted loan resolution date)

    preds_train = preds[preds["split"] == "train"].copy()
    lgd_col = pred_cols[0] if pred_cols else None

    if lgd_col and "orig_upb" in lgd_train.columns:
        # Merge to get UPB
        merged = preds_train.merge(
            lgd_train[["loan_seq_num", "orig_upb"]],
            on="loan_seq_num", how="inner"
        )
        if len(merged) > 0:
            merged["ecl"] = merged["actual_lgd"] * merged["orig_upb"]
            merged["ecl_pred"] = merged[lgd_col] * merged["orig_upb"]

            fig, axes = plt.subplots(1, 2, figsize=(13, 5))
            fig.suptitle("Illustrative ECL Distribution (LGD × Exposure)",
                         fontsize=13, color="#E2E8F0", fontweight="bold")

            axes[0].hist(merged["ecl"] / 1000, bins=40, color=DANGER, alpha=0.7,
                         edgecolor="none", density=True, label="Actual ECL")
            axes[0].hist(merged["ecl_pred"] / 1000, bins=40, color=WARN, alpha=0.55,
                         edgecolor="none", density=True, label=f"Predicted ECL ({lgd_col})")
            axes[0].set_xlabel("ECL ($000s)", fontsize=10)
            axes[0].set_ylabel("Density", fontsize=10)
            axes[0].set_title("ECL Distribution", color="#CBD5E1")
            axes[0].legend(fontsize=9)
            axes[0].grid(True, alpha=0.2)

            # Cumulative loss curve
            ecl_sorted = merged["ecl"].sort_values(ascending=False).reset_index(drop=True)
            cum_pct = (ecl_sorted.cumsum() / ecl_sorted.sum() * 100)
            loan_pct = np.linspace(0, 100, len(cum_pct))
            axes[1].plot(loan_pct, cum_pct, color=ACCENT, linewidth=2.5)
            axes[1].plot([0, 100], [0, 100], "k--", linewidth=1, alpha=0.5,
                         label="Proportional (random ordering)")
            axes[1].set_xlabel("Cumulative % of Loans (high-risk first)", fontsize=10)
            axes[1].set_ylabel("Cumulative % of Total Losses", fontsize=10)
            axes[1].set_title("Cumulative Loss Concentration", color="#CBD5E1")
            axes[1].legend(fontsize=9)
            axes[1].grid(True, alpha=0.2)

            plt.tight_layout()
            plt.savefig("data/figures/lgd_ecl_illustration.png",
                        dpi=150, bbox_inches="tight", facecolor="#0F1117")
            plt.show()

            top20 = merged.nlargest(int(len(merged)*0.2), "actual_lgd")["ecl"].sum()
            total = merged["ecl"].sum()
            print(f"Top 20% highest-LGD loans account for "
                  f"{top20/total*100:.1f}% of total losses")
        else:
            print("No matching loans between LGD predictions and training data.")
    else:
        print("orig_upb or LGD predictions not available.")
else:
    print("Run 04_lgd_models.py and 03_pd_ensemble.py first.")


## Summary

| Model | OOS RMSE | OOS R² | Key Strength |
|---|---|---|---|
| FRM (logit-OLS) | — | — | Interpretable, bounded predictions |
| Spline Regression | — | — | Non-linear without overfitting |
| Random Forest | — | — | Handles interactions well |
| XGBoost | — | — | Best predictive accuracy |

*(Fill in after running `04_lgd_models.py`)*

**Key takeaways:**
- LGD is harder to predict than PD — R² values of 0.3–0.5 are considered good in the literature.
- The FRM's bounded predictions are critical for regulatory capital calculations — you cannot have a negative LGD or LGD > 100%.
- The small sample size (~150 defaults in the sample data) means all metrics have wide confidence intervals. Pre-2010 crisis vintages are essential for robust LGD modelling.
